# Hidden Markov Models (HMMs) - Weather umbrella example

This notebook provides interactive widgets to explore probabilistic reasoning over time using the umbrella/weather example.

You can modify the model parameters and evidence sequences and see how beliefs evolve.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Textarea

states=['Rain','NoRain']

# Helper functions

Normalise a probability distribution

In [2]:
def normalize(d): 
    s=sum(d.values())
    return {k:v/s for k,v in d.items()}

Plots belief curves with probability labels and evidence markers.

In [3]:
def plot_history(history, evidence, title):
    t=list(range(len(history)))
    rain=[h['Rain'] for h in history]
    norain=[h['NoRain'] for h in history]
    plt.figure(figsize=(8,4))
    plt.plot(t,rain,label='Rain',marker='o')
    plt.plot(t,norain,label='NoRain',marker='o')
    plt.ylim(0,1.1)
    plt.xticks(t)
    for i,e in enumerate(evidence):
        if i+1 < len(history):
            plt.text(i+1,1.05,'Umbrella' if e else 'No Umbrella',ha='center',fontsize=9)
    for i,val in enumerate(rain): plt.text(i,val+0.03,str(round(val,2)),color='blue')
    for i,val in enumerate(norain): plt.text(i,val-0.07,str(round(val,2)),color='orange')
    plt.title(title)
    plt.xlabel('Time step')
    plt.ylabel('Probability')
    plt.legend()
    plt.show()

## Model Parameters (Adjustable)
This function defines the prior distribution, transition model, and sensor model. You can change their values directly from the sliders below. 

In [4]:
def get_model(prior_rain, trans_rr, trans_nr, sensor_r, sensor_nr):
    prior={'Rain':prior_rain,'NoRain':1-prior_rain}
    transition={('Rain','Rain'):trans_rr,('Rain','NoRain'):1-trans_rr,
                ('NoRain','Rain'):trans_nr,('NoRain','NoRain'):1-trans_nr}
    sensor={('Rain',True):sensor_r,('Rain',False):1-sensor_r,
            ('NoRain',True):sensor_nr,('NoRain',False):1-sensor_nr}
    return prior,transition,sensor

# 1. Inference

## 1.1. Filtering
Filtering incorporates past evidence to update beliefs about the latest state.

Enter any sequence of `True` and `False` to see the results of filtering. The default shows the example of the lecture.

In [5]:
def filtering(evidence_list, prior, transition, sensor):
    belief=prior.copy()
    history=[belief.copy()]
    for e in evidence_list:
        pred={s:0 for s in states}
        for s in states:
            for sp in states:
                pred[s]+=transition[(sp,s)]*belief[sp]
        updated={s:pred[s]*sensor[(s,e)] for s in states}
        belief=normalize(updated)
        history.append(belief.copy())
    return belief,history

In [ ]:
@interact(prior_rain=FloatSlider(min=0,max=1,step=0.01,value=0.5),
          trans_rr=FloatSlider(min=0,max=1,step=0.01,value=0.7),
          trans_nr=FloatSlider(min=0,max=1,step=0.01,value=0.3),
          sensor_r=FloatSlider(min=0,max=1,step=0.01,value=0.9),
          sensor_nr=FloatSlider(min=0,max=1,step=0.01,value=0.2),
          evidence_text=Textarea(value='True, True'))

def run_filter(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr,evidence_text):
    evidence=[eval(x.strip()) for x in evidence_text.split(',')]
    prior,transition,sensor=get_model(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr)
    _,hist=filtering(evidence,prior,transition,sensor)
    plot_history(hist,evidence,'Filtering with Evidence')

### Exercise

**Exercise 1**
Set the sensor accuracy to be almost perfect:

- `P(Umbrella | Rain) = 0.99`
- `P(Umbrella | NoRain) = 0.01`

Try several different evidence sequences.

- How does a nearly perfect sensor affect the belief updates?
- What happens when the evidence contradicts the transition model?

**Exercise 2 (To go further)**
Create a custom evidence sequence that makes your belief in Rain:

1. increase sharply, and then  
2. decrease sharply

Explain:
- Which evidence sequence you created
- Why it produces this up‑then‑down behavior

## 1.2. Prediction
Prediction does not use evidence. It answers: *Given my current belief (the prior), what will I believe after N future steps?*

In [ ]:
def predict(belief, steps, transition):
    b=belief.copy()
    history=[b.copy()]
    for _ in range(steps):
        new={s:0 for s in states}
        for s in states:
            for sp in states:
                new[s]+=transition[(sp,s)]*b[sp]
        b=new
        history.append(b.copy())
    return b,history

In [ ]:
@interact(prior_rain=FloatSlider(min=0,max=1,step=0.01,value=0.5),
          trans_rr=FloatSlider(min=0,max=1,step=0.01,value=0.7),
          trans_nr=FloatSlider(min=0,max=1,step=0.01,value=0.3),
          sensor_r=FloatSlider(min=0,max=1,step=0.01,value=0.9),
          sensor_nr=FloatSlider(min=0,max=1,step=0.01,value=0.2),
          steps=FloatSlider(min=1,max=10,step=1,value=3))

def run_pred(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr,steps):
    prior,transition,sensor=get_model(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr)
    belief=prior
    _,hist=predict(belief,int(steps),transition)
    plot_history(hist,[], 'Prediction for Future Steps')

interactive(children=(FloatSlider(value=0.5, description='prior_rain', max=1.0, step=0.01), FloatSlider(value=…

### Exercise

**Exercise 1**

Use the prediction widget with the following parameters:

- `P(Rain)=0.5`
- `P(Rain | Rain)=0.9`
- `P(Rain | NoRain)=0.1`
- Prediction steps = 10

Questions:
- What steady‑state (long‑term) probability does the belief converge to?
- Why does this convergence happen?

**Exercise 2**

Explain why prediction does not use any evidence and does not show “Umbrella / No Umbrella” text.


**Exercise 3 (To go further)** 

Use prediction steps = 5 and try these priors:

- `P(Rain)=0.1`
- `P(Rain)=0.5`
- `P(Rain)=0.9`

Questions:
- After 5 prediction steps, how different are the beliefs?
- Does the long‑term prediction depend strongly on the prior?
- What does this suggest about long‑term influence of initial uncertainty?

## 1.3. Smoothing
Smoothing uses all available evidence (past and future) to refine beliefs about earlier states.

In [11]:
def smoothing(evidence_list, prior, transition, sensor):
    _,fwd=filtering(evidence_list, prior, transition, sensor)
    T=len(evidence_list)
    bwd=[{'Rain':1,'NoRain':1}]
    for t in reversed(range(1,T+1)):
        new={} 
        for s in states:
            total=0
            for sp in states:
                total+=transition[(s,sp)]*sensor[(sp,evidence_list[t-1])]*bwd[-1][sp]
            new[s]=total
        new=normalize(new)
        bwd.append(new)
    bwd=bwd[::-1]
    smoothed=[]
    for t in range(T+1):
        prod={s:fwd[t][s]*bwd[t][s] for s in states}
        smoothed.append(normalize(prod))
    return smoothed

In [12]:
@interact(prior_rain=FloatSlider(min=0,max=1,step=0.01,value=0.5),
          trans_rr=FloatSlider(min=0,max=1,step=0.01,value=0.7),
          trans_nr=FloatSlider(min=0,max=1,step=0.01,value=0.3),
          sensor_r=FloatSlider(min=0,max=1,step=0.01,value=0.9),
          sensor_nr=FloatSlider(min=0,max=1,step=0.01,value=0.2),
          evidence_text=Textarea(value='True, True'))
def run_smooth(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr,evidence_text):
    evidence=[eval(x.strip()) for x in evidence_text.split(',')]
    prior,transition,sensor=get_model(prior_rain,trans_rr,trans_nr,sensor_r,sensor_nr)
    smoothed=smoothing(evidence,prior,transition,sensor)
    plot_history(smoothed,evidence,'Smoothed Beliefs')

interactive(children=(FloatSlider(value=0.5, description='prior_rain', max=1.0, step=0.01), FloatSlider(value=…

### Exercise

**Exercise 1**

Use smoothing on the evidence: `True, True, False`

Compare:
- Filtering results  
- Smoothing results  

Questions:
- Why do the smoothed beliefs for earlier time steps differ from filtered beliefs?
- What role does *future evidence* play?

**Exercise 2**

Use the evidence sequence: `False, False, True, True`

Questions:
- How does the presence of two “True” observations at the end affect your belief about the **first** state?
- Why does smoothing “pull back” information from the future?

**Exercise 3 (to go further)**

Try a noisy sequence: `False, True, False, True, False`

Questions:
- Does smoothing produce smoother, more stable curves compared with filtering?
- Why does smoothing help reduce the impact of random noise?